# AMAC-ECG — Stage 4 Final Evaluation + Decision Layer

This notebook uses one companion ZIP for the processed datasets, frozen folds, matching DS1 OOF predictions, and the Stage 4 runner.

## Locked experiment
- 5 model/imbalance configurations × RR enabled/disabled = 10 configurations.
- 5 random seeds per configuration = **50 DS2 final-evaluation runs**.
- LR unweighted, LR class-weighted, XGBoost class-weighted, CNN class-weighted, CNN focal loss.
- Same fixed DS1 patient-held-out fold (fold 0) across all seeds; random seeds vary independently.
- Stage-3-compatible leave-one-fold-out temperature calibration is reproduced using DS1 OOF predictions; calibration comparison is seed-0-specific.
- Class-wise N/S/V/F thresholds are selected from DS1 OOF only and frozen before DS2 evaluation.
- DS2 rejection/coverage-risk curves are descriptive; no threshold is selected using DS2.
- Patient-cluster bootstrap 95% confidence intervals (2,000 resamples).
- Q remains a model output but is excluded from headline N/S/V/F metrics.

**Important:** repository documentation records prior Stage-4 access to DS2. This run does not use DS2 for fitting, model selection, calibration fitting, or threshold tuning, but it does not claim DS2 is historically untouched/pristine.

**Do not use Run all.** Run cells in order. Stop if preflight fails.


## 1. Clone or refresh the GitHub repository

In [1]:
from pathlib import Path
import subprocess, os

REPO = Path("/content/AMAC-ECG")
if REPO.exists():
    print("Repository folder already exists; refreshing main branch.")
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/sanjanapriyadarsini/AMAC-ECG.git", str(REPO)],
        check=True
    )
os.chdir(REPO)
print("Working directory:", Path.cwd())
print("Repository files:", sorted(p.name for p in REPO.iterdir())[:20])

Working directory: /content/AMAC-ECG
Repository files: ['.git', '.gitignore', 'DATA_README.txt', 'IMPORTANT_BEFORE_RUNNING.txt', 'RUN_ORDER_CORRECTED.md', 'Stage-2-CNN', 'artifacts', 'requirements.txt', 'src', 'stage1.md', 'stage2.md', 'stage3.md']


## 2. Install project dependencies

In [2]:
%cd /content/AMAC-ECG
!pip -q install -r requirements.txt
print("Dependency installation cell completed.")

/content
Dependency installation cell completed.


## 3. Upload the complete Stage 4 companion package

Upload **`AMAC_ECG_Stage4_Final_Complete_Package.zip`**.

Do not upload `ds1.npz`, `ds2.npz`, `stage4.py`, folds, or OOF predictions separately; they are included in the package.


In [3]:
from google.colab import files
import zipfile, os
from pathlib import Path

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(zip_names) == 1, f"Upload exactly one companion ZIP; received: {list(uploaded)}"
package_name = zip_names[0]

with zipfile.ZipFile(package_name, "r") as zf:
    required_members = {
        "artifacts/ds1.npz",
        "artifacts/ds2.npz",
        "src/folds.json",
        "stage4.py",
        "STAGE4_README.md",
        "artifacts/preds/lr_none_rr_f0.npz",
        "artifacts/preds/lr_none_norr_f0.npz",
        "artifacts/preds/cnn_focal_rr_f4.npz",
        "artifacts/preds/cnn_focal_norr_f4.npz",
    }
    members = set(zf.namelist())
    missing = required_members - members
    assert not missing, f"Companion ZIP is incomplete: {sorted(missing)}"
    zf.extractall("/content/AMAC-ECG")

os.chdir("/content/AMAC-ECG")
print("Companion package extracted into:", Path.cwd())
print("DS1 exists:", Path("artifacts/ds1.npz").is_file())
print("DS2 exists:", Path("artifacts/ds2.npz").is_file())
print("Stage 4 runner exists:", Path("stage4.py").is_file())

Saving AMAC_ECG_Stage4_Final_Complete_Package.zip to AMAC_ECG_Stage4_Final_Complete_Package.zip
Companion package extracted into: /content/AMAC-ECG
DS1 exists: True
DS2 exists: True
Stage 4 runner exists: True


## 4. Runtime check

This cell checks the available accelerator. It does not open either dataset.

In [4]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable. CNN runs will be slower on CPU.")

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 5. DS1-only preflight — STOP if anything fails

This cell loads **DS1 only**. It verifies the frozen DS1 hash, patient-grouped folds, and all 50 DS1 OOF prediction files. It checks only that the DS2 file exists; it does not load DS2.


In [5]:
import hashlib, json, numpy as np, os
from pathlib import Path
import sys
sys.path.insert(0, "src")
import config as C
from run_grid import load_folds, file_sha256

ART = Path("artifacts")
assert (ART / "ds1.npz").is_file(), "Missing artifacts/ds1.npz"
assert (ART / "ds2.npz").is_file(), "Missing artifacts/ds2.npz (existence check only)"
assert (ART / "preds").is_dir(), "Missing artifacts/preds directory"

d1 = np.load(ART / "ds1.npz", allow_pickle=False)
assert d1["X"].shape == (53111, 216), d1["X"].shape
assert d1["rr"].shape == (53111, 4), d1["rr"].shape
assert all(k in d1.files for k in ["X", "rr", "y", "pid", "record_id"])

ds1_hash = file_sha256(ART / "ds1.npz")
folds = load_folds(d1["y"], d1["pid"], ds1_hash)
for fold_id, (tr, va) in enumerate(folds):
    assert set(d1["pid"][tr]).isdisjoint(set(d1["pid"][va])), f"Patient overlap in fold {fold_id}"
    print(f"Fold {fold_id}: train={len(tr)}, validation={len(va)}, patient-disjoint")

required = [
    ("lr", "none"), ("lr", "class_weight"), ("xgb", "class_weight"),
    ("cnn", "class_weight"), ("cnn", "focal")
]
checked = 0
for model, strategy in required:
    for rr_tag in ["rr", "norr"]:
        for fold_id, (_, va) in enumerate(folds):
            p = ART / "preds" / f"{model}_{strategy}_{rr_tag}_f{fold_id}.npz"
            assert p.is_file(), f"Missing OOF file: {p}"
            d = np.load(p, allow_pickle=False)
            assert str(d["dataset_hash"].item()) == ds1_hash, f"Hash mismatch: {p}"
            assert np.array_equal(d["val_idx"], va), f"Validation-index mismatch: {p}"
            assert np.array_equal(d["y_true"], d1["y"][va]), f"Label mismatch: {p}"
            assert np.array_equal(d["pid"], d1["pid"][va]), f"Patient-ID mismatch: {p}"
            checked += 1

print("\nPREFLIGHT PASSED")
print("DS1 SHA256:", ds1_hash)
print("Frozen folds:", len(folds))
print("OOF prediction files verified:", checked)
print("DS2 was not loaded by this preflight cell.")

folds: /content/AMAC-ECG/src/folds.json (frozen, made with sklearn 1.7.1)
Fold 0: train=45076, validation=8035, patient-disjoint
Fold 1: train=42142, validation=10969, patient-disjoint
Fold 2: train=39310, validation=13801, patient-disjoint
Fold 3: train=45511, validation=7600, patient-disjoint
Fold 4: train=40405, validation=12706, patient-disjoint

PREFLIGHT PASSED
DS1 SHA256: 74bfdda3fe8d2d73b13b00596271eb15418ee1bd13046e0aa7e42186eba54644
Frozen folds: 5
OOF prediction files verified: 50
DS2 was not loaded by this preflight cell.


## 6. Run Stage 4 final evaluation + decision layer

**This is the first cell that opens DS2.** It runs 10 configurations × 5 seeds = 50 final-evaluation runs.

It also:
- reproduces the Stage 3 leave-one-fold-out temperature procedure using DS1 OOF predictions;
- fits class-wise N/S/V/F thresholds on DS1 OOF only;
- freezes those thresholds and applies them to DS2;
- evaluates rejection, coverage, selective risk and selective Macro-F1;
- computes patient-cluster bootstrap 95% CIs;
- saves separate confusion matrices, PR curves, coverage-risk plots, comparison graphs, CSV/JSON and Markdown documentation.

CNN training may take time. Do not interrupt the runtime. Do not use Run all.


In [6]:
%cd /content/AMAC-ECG
!python stage4.py

/content
folds: /content/AMAC-ECG/src/folds.json (frozen, made with sklearn 1.7.1)

=== LR + Unweighted + RR ===
Seed-0 DS1 OOF temperature: 1.90046
  seed 0: fitting on DS1 and evaluating on DS2...
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
    Macro-F1=0.6040; Accuracy=0.9320; ECE=0.0186
  seed 1: fitting on DS1 and evaluating on DS2...
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
    Macro-F1=0.6040; Accuracy=0.9320; ECE=0.0186
  seed 2: fitting on DS1 and evaluating on DS2...
/usr/local/lib/python3.13/dist-packa

In [7]:
%cd /content/AMAC-ECG

import json
import time
import stage4 as s
from pathlib import Path

OUT = Path("artifacts/stage4")
result_file = OUT / "stage4_results.json"

assert result_file.exists(), (
    "Saved results not found. STOP and send screenshot."
)

result = json.loads(result_file.read_text())

assert len(result["configurations"]) == 10, (
    f"Expected 10 configurations, found {len(result['configurations'])}"
)

runs = sum(len(c["runs"]) for c in result["configurations"])
assert runs == 50, f"Expected 50 runs, found {runs}"

assert all("decision_layer" in c for c in result["configurations"])
assert all(
    "classwise_threshold_rejection_DS2" in r
    for c in result["configurations"]
    for r in c["runs"]
)

receipt = {
    "status": "completed",
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "elapsed_seconds": result["elapsed_seconds"],
    "ds1_sha256": result["dataset"]["ds1_sha256"],
    "ds2_sha256": result["dataset"]["ds2_sha256"],
    "seeds": result["seeds"],
    "configurations": [c["name"] for c in result["configurations"]],
    "runs_completed": runs,
    "ds2_usage_in_this_run": result["ds2_usage_in_this_run"],
    "historical_note": result["historical_note"],
}

(OUT / "RUN_RECEIPT.json").write_text(
    json.dumps(receipt, indent=2)
)

s.write_outputs(result)

print("RECOVERY COMPLETE")
print("Configurations:", len(result["configurations"]))
print("Runs recovered:", runs)
print("Threshold CSV:", (OUT / "classwise_thresholds_DS1_OOF.csv").exists())
print("Selective metrics CSV:", (OUT / "selective_metrics_by_seed.csv").exists())
print("Run receipt:", (OUT / "RUN_RECEIPT.json").exists())
print("Final report:", (OUT / "STAGE4_REPORT.md").exists())
print("PNG figures:", len(list((OUT / "figures").rglob("*.png"))))

/content
RECOVERY COMPLETE
Configurations: 10
Runs recovered: 50
Threshold CSV: True
Selective metrics CSV: True
Run receipt: True
Final report: True
PNG figures: 194


## 7. Verify generated deliverables

Confusion matrices and graphs are separate PNG files, not embedded in the Markdown report. This cell checks the core result files and decision-layer outputs.


In [8]:
from pathlib import Path
import json

OUT = Path("artifacts/stage4")
required = [
    "stage4_results.json",
    "stage4_metrics_by_seed.csv",
    "classwise_thresholds_DS1_OOF.csv",
    "selective_metrics_by_seed.csv",
    "STAGE4_REPORT.md",
    "RUN_RECEIPT.json",
]
for name in required:
    assert (OUT / name).is_file(), f"Missing deliverable: {name}"

receipt = json.loads((OUT / "RUN_RECEIPT.json").read_text())
assert receipt["status"] == "completed"
assert receipt["runs_completed"] == 50, receipt["runs_completed"]

pngs = list((OUT / "figures").rglob("*.png"))
pr_pngs = list((OUT / "figures" / "precision_recall").glob("*.png"))
reject_pngs = list((OUT / "figures" / "rejection").glob("*.png"))
assert len(pr_pngs) == 10, f"Expected 10 DS1 PR plots, found {len(pr_pngs)}"
assert len(reject_pngs) == 50, f"Expected 50 DS2 rejection plots, found {len(reject_pngs)}"

print("Status:", receipt["status"])
print("Runs completed:", receipt["runs_completed"])
print("Configurations:", len(receipt["configurations"]))
print("PNG figures:", len(pngs))
print("DS1 class-wise PR plots:", len(pr_pngs))
print("DS2 rejection plots:", len(reject_pngs))
print("CSV files:", len(list(OUT.glob("*.csv"))))
print("Report:", OUT / "STAGE4_REPORT.md")


Status: completed
Runs completed: 50
Configurations: 10
PNG figures: 194
DS1 class-wise PR plots: 10
DS2 rejection plots: 50
CSV files: 3
Report: artifacts/stage4/STAGE4_REPORT.md


## 8. Package all Stage 4 outputs for download

In [9]:
import shutil
from pathlib import Path

output_zip = Path("/content/AMAC_ECG_Stage4_Results.zip")
if output_zip.exists():
    output_zip.unlink()
shutil.make_archive(str(output_zip.with_suffix("")), "zip",
                    root_dir="/content/AMAC-ECG/artifacts", base_dir="stage4")

from google.colab import files
files.download(str(output_zip))
print("Stage 4 results package prepared.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Stage 4 results package prepared.
